# ICEF Energy Consumption Project

Hourly energy consumption analysis for 12 PJM Interconnection zones.  
Stack: **PostgreSQL** (window functions, CTEs, LATERAL) + **Python** (psycopg2, pandas, numpy).

**Authors:** Artem Avakov, Raffi Ordyan

## 0. Setup

Database connection parameters and utility functions.  
Update `DB_PARAMS` below if your credentials differ.

In [1]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from psycopg2 import sql

In [ ]:
DB_PARAMS = {
    'dbname': 'energy_project',
    'user': '',
    'password': '',
    'host': 'localhost',
    'port': '5432'
}

In [3]:
def get_conn():
    conn = psycopg2.connect(**DB_PARAMS)
    conn.autocommit = True
    return conn

def run_sql(query, params=None, fetch=False, conn=None):
    close_conn = False
    if conn is None:
        conn = get_conn()
        close_conn = True
    cur = conn.cursor()
    try:
        cur.execute(query, params)
        if fetch:
            cols = [desc[0] for desc in cur.description]
            rows = cur.fetchall()
            return pd.DataFrame(rows, columns=cols)
        return None
    finally:
        cur.close()
        if close_conn:
            conn.close()

def table_exists(table_name):
    q = """
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = 'public' AND table_name = %s
    ) AS exists_flag;
    """
    return bool(run_sql(q, (table_name,), fetch=True).iloc[0, 0])

def count_rows(table_name):
    return int(run_sql(f'SELECT COUNT(*) AS n FROM {table_name};', fetch=True).iloc[0, 0])

def print_table_status(table_name):
    if table_exists(table_name):
        print(f"{table_name}: exists, rows = {count_rows(table_name):,}")
    else:
        print(f"{table_name}: does NOT exist")

def show_other_sessions():
    q = """
    SELECT pid, usename, application_name, state,
           COALESCE(wait_event_type, '') AS wait_event_type,
           COALESCE(wait_event, '') AS wait_event,
           LEFT(query, 120) AS query_short
    FROM pg_stat_activity
    WHERE datname = current_database()
      AND pid <> pg_backend_pid()
    ORDER BY pid;
    """
    return run_sql(q, fetch=True)

def terminate_other_sessions_same_user(verbose=True):
    q = """
    SELECT pid
    FROM pg_stat_activity
    WHERE datname = current_database()
      AND pid <> pg_backend_pid()
      AND usename = current_user;
    """
    pids_df = run_sql(q, fetch=True)
    if pids_df.empty:
        if verbose:
            print("No other sessions of the current user were found.")
        return True

    ok = True
    for pid in pids_df['pid'].tolist():
        try:
            run_sql("SELECT pg_terminate_backend(%s);", (int(pid),), fetch=False)
            if verbose:
                print(f"Terminated backend pid={pid}")
        except Exception as e:
            ok = False
            if verbose:
                print(f"Could not terminate pid={pid}: {e}")
    return ok

def drop_tables_fast(table_names, lock_timeout='2s', statement_timeout='30s', conn=None):
    close_conn = False
    if conn is None:
        conn = get_conn()
        close_conn = True
    cur = conn.cursor()
    dropped = []
    failed = []
    try:
        cur.execute(f"SET lock_timeout = '{lock_timeout}';")
        cur.execute(f"SET statement_timeout = '{statement_timeout}';")
        for table_name in table_names:
            try:
                cur.execute(sql.SQL("DROP TABLE IF EXISTS {} CASCADE;").format(sql.Identifier(table_name)))
                dropped.append(table_name)
            except Exception as e:
                failed.append((table_name, str(e)))
                conn.rollback()
                cur.execute(f"SET lock_timeout = '{lock_timeout}';")
                cur.execute(f"SET statement_timeout = '{statement_timeout}';")
    finally:
        cur.close()
        if close_conn:
            conn.close()
    return dropped, failed

def recreate_energy_load():
    tables_to_drop = [
        'energy_consolidated',
        'prediction_features',
        'zone_neighbor_correlation',
        'neighbor_map',
        'zone_correlations',
        'daily_clusters',
        'daily_profiles',
        'volatility_regimes',
        'volatility_transitions',
        'rolling_stats',
        'temporal_features',
        'gap_info',
        'gap_summary',
        'missing_hours',
        'energy_load',
    ]

    sessions_df = show_other_sessions()
    if not sessions_df.empty:
        print("Other active sessions in the same database were found:")
        display(sessions_df)
        print("Attempting to terminate other sessions of the current user...")

    terminate_other_sessions_same_user(verbose=True)

    dropped, failed = drop_tables_fast(tables_to_drop)

    print(f"Dropped successfully: {len(dropped)} tables")
    if failed:
        failed_df = pd.DataFrame(failed, columns=['table_name', 'error'])
        print("Could not drop some tables. Most likely there is still a lock from another open notebook / VS Code / psql session.")
        display(failed_df)
        raise RuntimeError(
            "Some tables could not be dropped because they are locked. "
            "Close other Jupyter / VS Code / psql sessions connected to energy_project, "
            "then run recreate_energy_load() again."
        )

    run_sql("""
    CREATE TABLE energy_load (
        zone VARCHAR(15) NOT NULL,
        datetime TIMESTAMP NOT NULL,
        load_mw DOUBLE PRECISION,
        is_imputed SMALLINT DEFAULT 0,
        PRIMARY KEY (zone, datetime)
    );
    """)

In [4]:
test_df = run_sql("SELECT current_database() AS db, current_user AS usr;", fetch=True)
print(test_df)

               db     usr
0  energy_project  avakov


## 1. Load raw data

Locate the 12 CSV files and load them into the `energy_load` table in PostgreSQL.  
Each row represents one zone + one hour + one load value (MW).

In [5]:
zone_files = {
    'AEP':      'AEP_hourly.csv',
    'COMED':    'COMED_hourly.csv',
    'DAYTON':   'DAYTON_hourly.csv',
    'DEOK':     'DEOK_hourly.csv',
    'DOM':      'DOM_hourly.csv',
    'DUQ':      'DUQ_hourly.csv',
    'EKPC':     'EKPC_hourly.csv',
    'FE':       'FE_hourly.csv',
    'NI':       'NI_hourly.csv',
    'PJME':     'PJME_hourly.csv',
    'PJMW':     'PJMW_hourly.csv',
    'PJM_Load': 'PJM_Load_hourly.csv',
}

def find_data_dir(required_files):
    candidates = [
        Path.cwd(),
        Path.cwd() / 'data',
        Path.home() / 'Downloads',
        Path('/mnt/data'),
    ]
    for folder in candidates:
        if all((folder / fname).exists() for fname in required_files.values()):
            return folder

    print("Could not find a directory containing all required CSV files.")
    for folder in candidates:
        existing = [fname for fname in required_files.values() if (folder / fname).exists()]
        if existing:
            print(f"Partial match in {folder}: {existing}")
    raise FileNotFoundError("Update the search paths or move the CSV files into one folder.")

DATA_DIR = find_data_dir(zone_files)
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/avakov/Downloads


### 1.1 Recreate the raw table

Drop all project tables and recreate `energy_load` with a composite primary key `(zone, datetime)`.

In [6]:
recreate_energy_load()
print("energy_load recreated.")

No other sessions of the current user were found.
Dropped successfully: 15 tables
energy_load recreated.


In [7]:
conn = get_conn()
cur = conn.cursor()

for zone_name, filename in zone_files.items():
    filepath = Path(DATA_DIR) / filename
    df = pd.read_csv(filepath)

    if df.shape[1] < 2:
        raise ValueError(f"{filename} has fewer than 2 columns.")

    dt_col = df.columns[0]
    load_col = df.columns[1]

    df = df.rename(columns={dt_col: 'datetime', load_col: 'load_mw'})
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['load_mw'] = pd.to_numeric(df['load_mw'], errors='coerce')
    df['zone'] = zone_name
    df['is_imputed'] = 0
    df = df[['zone', 'datetime', 'load_mw', 'is_imputed']].dropna(subset=['datetime'])

    tuples = list(df.itertuples(index=False, name=None))

    execute_values(
        cur,
        """
        INSERT INTO energy_load (zone, datetime, load_mw, is_imputed)
        VALUES %s
        ON CONFLICT (zone, datetime) DO NOTHING
        """,
        tuples,
        page_size=10000
    )
    print(f"Loaded {zone_name}: {len(df):,} rows")

cur.close()
conn.close()
print_table_status('energy_load')

Loaded AEP: 121,273 rows
Loaded COMED: 66,497 rows
Loaded DAYTON: 121,275 rows
Loaded DEOK: 57,739 rows
Loaded DOM: 116,189 rows
Loaded DUQ: 119,068 rows
Loaded EKPC: 45,334 rows
Loaded FE: 62,874 rows
Loaded NI: 58,450 rows
Loaded PJME: 145,366 rows
Loaded PJMW: 143,206 rows
Loaded PJM_Load: 32,896 rows
energy_load: exists, rows = 1,090,127


In [8]:
run_sql("CREATE INDEX IF NOT EXISTS idx_energy_load_zone_dt ON energy_load(zone, datetime);")
run_sql("ANALYZE energy_load;")
run_sql("""
SELECT zone, COUNT(*) AS n_rows, MIN(datetime) AS first_ts, MAX(datetime) AS last_ts
FROM energy_load
GROUP BY zone
ORDER BY zone;
""", fetch=True)

,zone,n_rows,first_ts,last_ts
0,AEP,121269,2004-10-01 01:00:00,2018-08-03
1,COMED,66493,2011-01-01 01:00:00,2018-08-03
2,DAYTON,121271,2004-10-01 01:00:00,2018-08-03
3,DEOK,57735,2012-01-01 01:00:00,2018-08-03
4,DOM,116185,2005-05-01 01:00:00,2018-08-03
5,DUQ,119064,2005-01-01 01:00:00,2018-08-03
6,EKPC,45330,2013-06-01 01:00:00,2018-08-03
7,FE,62870,2011-06-01 01:00:00,2018-08-03
8,NI,58450,2004-05-01 01:00:00,2011-01-01
9,PJME,145362,2002-01-01 01:00:00,2018-08-03


## 2. Step 1 - Missing-hour analysis

Identify all missing hours in each zone's time series using `generate_series`.  
Group consecutive gaps via the gaps-and-islands technique (`ROW_NUMBER`).  
Output: `missing_hours` (individual missing timestamps) and `gap_summary` (per-zone statistics).

In [9]:
run_sql("""
DROP TABLE IF EXISTS missing_hours CASCADE;
CREATE TABLE missing_hours AS
WITH zone_bounds AS (
    SELECT zone, MIN(datetime) AS ts_min, MAX(datetime) AS ts_max
    FROM energy_load
    GROUP BY zone
),
expected_hours AS (
    SELECT zb.zone, gs AS datetime
    FROM zone_bounds zb
    CROSS JOIN LATERAL generate_series(zb.ts_min, zb.ts_max, INTERVAL '1 hour') AS gs
)
SELECT eh.zone, eh.datetime
FROM expected_hours eh
LEFT JOIN energy_load el
    ON el.zone = eh.zone AND el.datetime = eh.datetime
WHERE el.datetime IS NULL;
""")

run_sql("""
DROP TABLE IF EXISTS gap_summary CASCADE;
CREATE TABLE gap_summary AS
WITH grouped AS (
    SELECT
        zone,
        datetime,
        datetime - (ROW_NUMBER() OVER (PARTITION BY zone ORDER BY datetime) * INTERVAL '1 hour') AS grp
    FROM missing_hours
),
per_gap AS (
    SELECT zone,
           MIN(datetime) AS gap_start,
           MAX(datetime) AS gap_end,
           COUNT(*) AS gap_length_hours
    FROM grouped
    GROUP BY zone, grp
)
SELECT
    zone,
    COUNT(*) AS number_of_gaps,
    COALESCE(SUM(gap_length_hours), 0) AS total_missing_hours,
    COALESCE(MAX(gap_length_hours), 0) AS longest_gap_hours
FROM per_gap
GROUP BY zone;
""")

print_table_status('missing_hours')
print_table_status('gap_summary')

missing_hours: exists, rows = 217
gap_summary: exists, rows = 12


In [10]:
run_sql("SELECT * FROM gap_summary ORDER BY zone;", fetch=True)

,zone,number_of_gaps,total_missing_hours,longest_gap_hours
0,AEP,27,27,1
1,COMED,11,11,1
2,DAYTON,25,25,1
3,DEOK,9,9,1
4,DOM,23,23,1
5,DUQ,24,24,1
6,EKPC,6,6,1
7,FE,10,10,1
8,NI,14,14,1
9,PJME,30,30,1


## 3. Step 2 - gap filling

Three strategies based on gap length:
- **≤ 3 hours:** linear interpolation between nearest known neighbors (LATERAL)
- **4–24 hours:** same-hour previous-day value with geometric trend adjustment
- **> 24 hours:** quadratic polynomial interpolation (Python)

First -  insert placeholder rows with `NULL` for all missing hours, then build `gap_info` with gap lengths.


In [11]:
run_sql("""
WITH zone_bounds AS (
    SELECT zone, MIN(datetime) AS ts_min, MAX(datetime) AS ts_max
    FROM energy_load
    GROUP BY zone
),
expected_hours AS (
    SELECT zb.zone, gs AS datetime
    FROM zone_bounds zb
    CROSS JOIN LATERAL generate_series(zb.ts_min, zb.ts_max, INTERVAL '1 hour') AS gs
)
INSERT INTO energy_load (zone, datetime, load_mw, is_imputed)
SELECT eh.zone, eh.datetime, NULL, 0
FROM expected_hours eh
LEFT JOIN energy_load el
    ON el.zone = eh.zone AND el.datetime = eh.datetime
WHERE el.datetime IS NULL;
""")

run_sql("""
DROP TABLE IF EXISTS gap_info CASCADE;
CREATE TABLE gap_info AS
WITH null_rows AS (
    SELECT zone, datetime
    FROM energy_load
    WHERE load_mw IS NULL
),
grouped AS (
    SELECT
        zone,
        datetime,
        datetime - (ROW_NUMBER() OVER (PARTITION BY zone ORDER BY datetime) * INTERVAL '1 hour') AS grp
    FROM null_rows
),
per_gap AS (
    SELECT zone,
           grp,
           MIN(datetime) AS gap_start,
           MAX(datetime) AS gap_end,
           COUNT(*) AS gap_length
    FROM grouped
    GROUP BY zone, grp
)
SELECT g.zone, g.datetime, pg.gap_start, pg.gap_end, pg.gap_length
FROM grouped g
JOIN per_gap pg
  ON g.zone = pg.zone AND g.grp = pg.grp;
""")

print_table_status('gap_info')

gap_info: exists, rows = 217


### 3.1 Short gaps (≤ 3 hours) - Linear interpolation

For each missing row, find the nearest non-NULL points before and after using `CROSS JOIN LATERAL`, then linearly interpolate

In [12]:
run_sql("""
UPDATE energy_load el
SET load_mw = filled.interpolated_val,
    is_imputed = 1
FROM (
    SELECT
        gi.zone,
        gi.datetime,
        prev.load_mw
        + (nxt.load_mw - prev.load_mw)
          * EXTRACT(EPOCH FROM (gi.datetime - prev.datetime))
          / NULLIF(EXTRACT(EPOCH FROM (nxt.datetime - prev.datetime)), 0)
        AS interpolated_val
    FROM gap_info gi
    CROSS JOIN LATERAL (
        SELECT e.datetime, e.load_mw
        FROM energy_load e
        WHERE e.zone = gi.zone
          AND e.datetime < gi.datetime
          AND e.load_mw IS NOT NULL
        ORDER BY e.datetime DESC
        LIMIT 1
    ) prev
    CROSS JOIN LATERAL (
        SELECT e.datetime, e.load_mw
        FROM energy_load e
        WHERE e.zone = gi.zone
          AND e.datetime > gi.datetime
          AND e.load_mw IS NOT NULL
        ORDER BY e.datetime ASC
        LIMIT 1
    ) nxt
    WHERE gi.gap_length <= 3
) filled
WHERE el.zone = filled.zone
  AND el.datetime = filled.datetime
  AND el.load_mw IS NULL;
""")

run_sql("ANALYZE energy_load;")
print("Step 3.1 done — short gaps filled with linear interpolation.")

Step 3.1 done — short gaps filled with linear interpolation.


### 3.2 Medium gaps (4-24 hours) - Previous-day + trend

Use the same hour from the previous day as a base value, adjusted by a geometric trend factor: `√(next_known / prev_known)`.

In [13]:
run_sql("""
UPDATE energy_load el
SET load_mw = filled.filled_val,
    is_imputed = 1
FROM (
    SELECT
        gi.zone,
        gi.datetime,
        prev_day.load_mw *
        COALESCE(
            SQRT(NULLIF(next_known.load_mw, 0) / NULLIF(prev_known.load_mw, 0)),
            1.0
        ) AS filled_val
    FROM gap_info gi
    LEFT JOIN energy_load prev_day
        ON prev_day.zone = gi.zone
       AND prev_day.datetime = gi.datetime - INTERVAL '1 day'
       AND prev_day.load_mw IS NOT NULL
    LEFT JOIN LATERAL (
        SELECT load_mw
        FROM energy_load e1
        WHERE e1.zone = gi.zone
          AND e1.datetime < gi.datetime
          AND e1.load_mw IS NOT NULL
        ORDER BY e1.datetime DESC
        LIMIT 1
    ) prev_known ON TRUE
    LEFT JOIN LATERAL (
        SELECT load_mw
        FROM energy_load e2
        WHERE e2.zone = gi.zone
          AND e2.datetime > gi.datetime
          AND e2.load_mw IS NOT NULL
        ORDER BY e2.datetime ASC
        LIMIT 1
    ) next_known ON TRUE
    WHERE gi.gap_length BETWEEN 4 AND 24
      AND prev_day.load_mw IS NOT NULL
) filled
WHERE el.zone = filled.zone
  AND el.datetime = filled.datetime
  AND el.load_mw IS NULL;
""")

run_sql("ANALYZE energy_load;")

### 3.3 Long gaps (> 24 hours) - Polynomial interpolation

Fit a degree-2 polynomial through the 3 nearest known points on each side. Done in Python (`numpy.polyfit`).

In [14]:
def quadratic_fill_for_zone(df_zone, long_gap_times):
    df_zone = df_zone.sort_values('datetime').copy()
    known = df_zone[df_zone['load_mw'].notna()].copy()
    if known.shape[0] < 5:
        return {}

    known['t'] = (known['datetime'] - known['datetime'].min()).dt.total_seconds() / 3600.0
    result = {}

    for ts in long_gap_times:
        target_t = (ts - known['datetime'].min()).total_seconds() / 3600.0
        left = known[known['datetime'] < ts].tail(3)
        right = known[known['datetime'] > ts].head(3)
        pts = pd.concat([left, right]).drop_duplicates(subset=['datetime'])

        if pts.shape[0] < 3:
            continue

        x = pts['t'].to_numpy(dtype=float)
        y = pts['load_mw'].to_numpy(dtype=float)

        try:
            coeffs = np.polyfit(x, y, deg=2 if pts.shape[0] >= 3 else 1)
            pred = np.polyval(coeffs, target_t)
            if np.isfinite(pred):
                result[ts] = float(pred)
        except Exception:
            continue

    return result

long_gap_df = run_sql("""
SELECT zone, datetime
FROM gap_info
WHERE gap_length > 24
ORDER BY zone, datetime;
""", fetch=True)

if long_gap_df.empty:
    print("No long gaps (>24h) detected, polynomial interpolation step is effectively empty.")
else:
    energy_df = run_sql("""
    SELECT zone, datetime, load_mw
    FROM energy_load
    ORDER BY zone, datetime;
    """, fetch=True)
    energy_df['datetime'] = pd.to_datetime(energy_df['datetime'])

    updates = []
    for zone, sub in long_gap_df.groupby('zone'):
        ts_list = pd.to_datetime(sub['datetime']).tolist()
        zone_df = energy_df[energy_df['zone'] == zone].copy()
        filled_map = quadratic_fill_for_zone(zone_df, ts_list)
        for ts, val in filled_map.items():
            updates.append((val, 1, zone, ts.to_pydatetime()))

    if updates:
        conn = get_conn()
        cur = conn.cursor()
        execute_values(
            cur,
            """
            UPDATE energy_load AS el
            SET load_mw = data.val,
                is_imputed = data.is_imputed
            FROM (VALUES %s) AS data(val, is_imputed, zone, datetime)
            WHERE el.zone = data.zone
              AND el.datetime = data.datetime
              AND el.load_mw IS NULL
            """,
            updates,
            page_size=1000
        )
        cur.close()
        conn.close()
        print(f"Polynomial interpolation applied to {len(updates):,} long-gap rows.")
    else:
        print("Long gaps exist, but no rows were updated by quadratic interpolation.")

No long gaps (>24h) detected, polynomial interpolation step is effectively empty.


In [15]:
run_sql("""
SELECT COUNT(*) AS remaining_nulls
FROM energy_load
WHERE load_mw IS NULL;
""", fetch=True)

,remaining_nulls
0,0


## 4. Step 3 - Temporal features

Extract calendar and behavioral features into `temporal_features`:  
hour, day of week, weekend flag, business hour flag, mounth, season, and a peak indicator based on the zone's 75th percentile during business hours

In [16]:
run_sql("""
DROP TABLE IF EXISTS temporal_features CASCADE;
CREATE TABLE temporal_features AS
WITH p75 AS (
    SELECT
        zone,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY load_mw) AS p75_load
    FROM energy_load
    WHERE EXTRACT(DOW FROM datetime) NOT IN (0, 6)
      AND EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 17
    GROUP BY zone
)
SELECT
    el.zone,
    el.datetime,
    EXTRACT(HOUR FROM el.datetime)::SMALLINT AS hour_of_day,
    EXTRACT(DOW  FROM el.datetime)::SMALLINT AS day_of_week,
    CASE WHEN EXTRACT(DOW FROM el.datetime) IN (0, 6) THEN 1 ELSE 0 END::SMALLINT AS is_weekend,
    CASE
        WHEN EXTRACT(DOW FROM el.datetime) NOT IN (0, 6)
         AND EXTRACT(HOUR FROM el.datetime) BETWEEN 8 AND 17
        THEN 1 ELSE 0
    END::SMALLINT AS is_business_hour,
    EXTRACT(MONTH FROM el.datetime)::SMALLINT AS month,
    CASE
        WHEN EXTRACT(MONTH FROM el.datetime) IN (12, 1, 2) THEN 'winter'
        WHEN EXTRACT(MONTH FROM el.datetime) IN (6, 7, 8)  THEN 'summer'
        ELSE 'transition'
    END AS season,
    CASE
        WHEN EXTRACT(DOW FROM el.datetime) NOT IN (0, 6)
         AND EXTRACT(HOUR FROM el.datetime) BETWEEN 8 AND 17
         AND el.load_mw >= p75.p75_load
        THEN 1 ELSE 0
    END::SMALLINT AS is_peak
FROM energy_load el
LEFT JOIN p75
  ON el.zone = p75.zone;
""")
print_table_status('temporal_features')

temporal_features: exists, rows = 1,090,344


## 5. Step 4 - Rolling statistics

Compute rolling mean and standard deviation at four horizons (6h, 24h, 72h, 168h) per zone using named `WINDOW` clauses.

In [17]:
run_sql("""
DROP TABLE IF EXISTS rolling_stats CASCADE;
CREATE TABLE rolling_stats AS
SELECT
    zone,
    datetime,
    AVG(load_mw)    OVER w6   AS rolling_mean_6h,
    STDDEV(load_mw) OVER w6   AS rolling_std_6h,
    AVG(load_mw)    OVER w24  AS rolling_mean_24h,
    STDDEV(load_mw) OVER w24  AS rolling_std_24h,
    AVG(load_mw)    OVER w72  AS rolling_mean_72h,
    STDDEV(load_mw) OVER w72  AS rolling_std_72h,
    AVG(load_mw)    OVER w168 AS rolling_mean_168h,
    STDDEV(load_mw) OVER w168 AS rolling_std_168h
FROM energy_load
WINDOW
    w6   AS (PARTITION BY zone ORDER BY datetime ROWS BETWEEN 5 PRECEDING   AND CURRENT ROW),
    w24  AS (PARTITION BY zone ORDER BY datetime ROWS BETWEEN 23 PRECEDING  AND CURRENT ROW),
    w72  AS (PARTITION BY zone ORDER BY datetime ROWS BETWEEN 71 PRECEDING  AND CURRENT ROW),
    w168 AS (PARTITION BY zone ORDER BY datetime ROWS BETWEEN 167 PRECEDING AND CURRENT ROW);
""")
print_table_status('rolling_stats')

rolling_stats: exists, rows = 1,090,344


## 6. Step 5 - Volatility regimes

Compute rolling 24-hour volatility (STDDEV of hourly changes), average it per day, then classify each day into **low** / **normal** / **high** using `NTILE(5)`.  
Also build a regime transition matrix.

In [18]:
run_sql("""
DROP TABLE IF EXISTS volatility_regimes CASCADE;
CREATE TABLE volatility_regimes AS
WITH hourly_changes AS (
    SELECT
        zone,
        datetime,
        load_mw - LAG(load_mw) OVER (PARTITION BY zone ORDER BY datetime) AS hourly_change
    FROM energy_load
),
rolling_vol AS (
    SELECT
        zone,
        datetime,
        STDDEV(hourly_change) OVER (
            PARTITION BY zone
            ORDER BY datetime
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        ) AS volatility_24h
    FROM hourly_changes
),
daily_vol AS (
    SELECT
        zone,
        datetime::date AS day,
        AVG(volatility_24h) AS daily_volatility
    FROM rolling_vol
    WHERE volatility_24h IS NOT NULL
    GROUP BY zone, datetime::date
),
bucketed AS (
    SELECT
        zone,
        day,
        daily_volatility,
        NTILE(5) OVER (PARTITION BY zone ORDER BY daily_volatility) AS quintile
    FROM daily_vol
)
SELECT
    zone,
    day,
    daily_volatility,
    CASE
        WHEN quintile = 1 THEN 'low'
        WHEN quintile = 5 THEN 'high'
        ELSE 'normal'
    END AS volatility_regime
FROM bucketed;
""")

run_sql("""
DROP TABLE IF EXISTS volatility_transitions CASCADE;
CREATE TABLE volatility_transitions AS
WITH ordered AS (
    SELECT
        zone,
        day,
        volatility_regime,
        LAG(volatility_regime) OVER (PARTITION BY zone ORDER BY day) AS prev_regime
    FROM volatility_regimes
)
SELECT
    zone,
    prev_regime,
    volatility_regime AS next_regime,
    COUNT(*) AS transition_count
FROM ordered
WHERE prev_regime IS NOT NULL
GROUP BY zone, prev_regime, volatility_regime;
""")

print_table_status('volatility_regimes')
print_table_status('volatility_transitions')

volatility_regimes: exists, rows = 45,443
volatility_transitions: exists, rows = 103


## 7. Step 6 - Daily profiles & pattern clustering

Compute 5 daily shape features (morning peak, evening peak, overnight base, ramp-up rate, ramp-down rate), then assign each day to one of 5 pre-defined consumption patterns via Euclidean distance to fixed centroids in z-normalized space.

In [19]:
run_sql("""
DROP TABLE IF EXISTS daily_profiles CASCADE;
CREATE TABLE daily_profiles AS
WITH hourly AS (
    SELECT
        zone,
        datetime::date AS day,
        EXTRACT(HOUR FROM datetime) AS hr,
        load_mw,
        load_mw - LAG(load_mw) OVER (PARTITION BY zone ORDER BY datetime) AS hourly_delta
    FROM energy_load
)
SELECT
    zone,
    day,
    MAX(CASE WHEN hr BETWEEN 6 AND 10  THEN load_mw END)      AS morning_peak,
    MAX(CASE WHEN hr BETWEEN 17 AND 21 THEN load_mw END)      AS evening_peak,
    AVG(CASE WHEN hr BETWEEN 0 AND 5   THEN load_mw END)      AS overnight_base,
    AVG(CASE WHEN hr BETWEEN 6 AND 9   THEN hourly_delta END) AS ramp_up_rate,
    AVG(CASE WHEN hr BETWEEN 19 AND 22 THEN hourly_delta END) AS ramp_down_rate
FROM hourly
GROUP BY zone, day;
""")

run_sql("""
DROP TABLE IF EXISTS daily_clusters CASCADE;
CREATE TABLE daily_clusters AS
WITH stats AS (
    SELECT
        zone,
        AVG(morning_peak) AS avg_mp, STDDEV(morning_peak) AS sd_mp,
        AVG(evening_peak) AS avg_ep, STDDEV(evening_peak) AS sd_ep,
        AVG(overnight_base) AS avg_ob, STDDEV(overnight_base) AS sd_ob,
        AVG(ramp_up_rate) AS avg_ru, STDDEV(ramp_up_rate) AS sd_ru,
        AVG(ramp_down_rate) AS avg_rd, STDDEV(ramp_down_rate) AS sd_rd
    FROM daily_profiles
    GROUP BY zone
),
normed AS (
    SELECT
        dp.zone,
        dp.day,
        (dp.morning_peak   - s.avg_mp) / NULLIF(s.sd_mp, 0) AS n_mp,
        (dp.evening_peak   - s.avg_ep) / NULLIF(s.sd_ep, 0) AS n_ep,
        (dp.overnight_base - s.avg_ob) / NULLIF(s.sd_ob, 0) AS n_ob,
        (dp.ramp_up_rate   - s.avg_ru) / NULLIF(s.sd_ru, 0) AS n_ru,
        (dp.ramp_down_rate - s.avg_rd) / NULLIF(s.sd_rd, 0) AS n_rd
    FROM daily_profiles dp
    JOIN stats s ON dp.zone = s.zone
),
distances AS (
    SELECT
        zone,
        day,
        SQRT(POWER(n_mp-(-1.0),2)+POWER(n_ep-(-1.0),2)+POWER(n_ob-(-0.5),2)+POWER(n_ru-0.0,2)+POWER(n_rd-0.0,2)) AS d_flat_low,
        SQRT(POWER(n_mp-( 1.0),2)+POWER(n_ep-( 0.0),2)+POWER(n_ob-( 0.0),2)+POWER(n_ru-1.0,2)+POWER(n_rd-0.0,2)) AS d_morning_dom,
        SQRT(POWER(n_mp-( 0.0),2)+POWER(n_ep-( 1.0),2)+POWER(n_ob-( 0.0),2)+POWER(n_ru-0.0,2)+POWER(n_rd-(-1.0),2)) AS d_evening_dom,
        SQRT(POWER(n_mp-( 1.0),2)+POWER(n_ep-( 1.0),2)+POWER(n_ob-( 0.0),2)+POWER(n_ru-0.5,2)+POWER(n_rd-(-0.5),2)) AS d_double_peak,
        SQRT(POWER(n_mp-( 0.0),2)+POWER(n_ep-( 0.0),2)+POWER(n_ob-( 1.5),2)+POWER(n_ru-(-0.5),2)+POWER(n_rd-0.5,2)) AS d_high_base
    FROM normed
)
SELECT
    zone,
    day,
    CASE
        WHEN d_flat_low    <= d_morning_dom AND d_flat_low    <= d_evening_dom AND d_flat_low    <= d_double_peak AND d_flat_low    <= d_high_base THEN 'flat_low'
        WHEN d_morning_dom <= d_flat_low    AND d_morning_dom <= d_evening_dom AND d_morning_dom <= d_double_peak AND d_morning_dom <= d_high_base THEN 'morning_dom'
        WHEN d_evening_dom <= d_flat_low    AND d_evening_dom <= d_morning_dom AND d_evening_dom <= d_double_peak AND d_evening_dom <= d_high_base THEN 'evening_dom'
        WHEN d_double_peak <= d_flat_low    AND d_double_peak <= d_morning_dom AND d_double_peak <= d_evening_dom AND d_double_peak <= d_high_base THEN 'double_peak'
        ELSE 'high_base'
    END AS consumption_pattern
FROM distances;
""")

print_table_status('daily_profiles')
print_table_status('daily_clusters')

daily_profiles: exists, rows = 45,443
daily_clusters: exists, rows = 45,443


## 8. Step 7 - Inter-zone correlations

1. Compute overall `CORR()` for all 66 zone pairs → pick the most correlated neighbor per zone (`neighbor_map`).  
2. Compute rolling 24h correlation only for the 12 selected neighbor pairs.  
3. Flag correlation breakdowns (< 0.5) as potential grid anomalies.

In [20]:
run_sql("""
DROP TABLE IF EXISTS zone_correlations CASCADE;
DROP TABLE IF EXISTS neighbor_map CASCADE;
DROP TABLE IF EXISTS zone_neighbor_correlation CASCADE;
""")

run_sql("""
CREATE TABLE neighbor_map AS
WITH pair_corr AS (
    SELECT
        a.zone  AS zone_a,
        b.zone  AS zone_b,
        CORR(a.load_mw, b.load_mw) AS avg_corr
    FROM energy_load a
    JOIN energy_load b
      ON a.datetime = b.datetime
     AND a.zone < b.zone
    GROUP BY a.zone, b.zone
),
long_form AS (
    SELECT zone_a AS zone, zone_b AS neighbor_zone, avg_corr FROM pair_corr
    UNION ALL
    SELECT zone_b AS zone, zone_a AS neighbor_zone, avg_corr FROM pair_corr
),
ranked AS (
    SELECT
        zone,
        neighbor_zone,
        avg_corr,
        ROW_NUMBER() OVER (PARTITION BY zone ORDER BY avg_corr DESC NULLS LAST) AS rn
    FROM long_form
)
SELECT zone, neighbor_zone, avg_corr
FROM ranked
WHERE rn = 1;
""")
print("neighbor_map created.")
run_sql("SELECT * FROM neighbor_map ORDER BY zone;", fetch=True)

neighbor_map created.


,zone,neighbor_zone,avg_corr
0,AEP,DAYTON,0.943818
1,COMED,FE,0.903888
2,DAYTON,FE,0.965563
3,DEOK,DAYTON,0.962329
4,DOM,PJME,0.927533
5,DUQ,FE,0.929663
6,EKPC,AEP,0.892500
7,FE,DAYTON,0.965563
8,NI,DUQ,0.899570
9,PJME,DUQ,0.929177


In [21]:
run_sql("""
CREATE TABLE zone_correlations AS
SELECT
    a.zone  AS zone_a,
    nm.neighbor_zone AS zone_b,
    a.datetime,
    CORR(a.load_mw, b.load_mw) OVER (
        PARTITION BY a.zone
        ORDER BY a.datetime
        ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
    ) AS correlation_24h
FROM energy_load a
JOIN neighbor_map nm
  ON a.zone = nm.zone
JOIN energy_load b
  ON b.zone  = nm.neighbor_zone
 AND b.datetime = a.datetime;
""")
print("zone_correlations created (neighbor pairs only).")

run_sql("""
CREATE TABLE zone_neighbor_correlation AS
SELECT
    zone_a AS zone,
    datetime,
    zone_b AS neighbor_zone,
    correlation_24h,
    CASE WHEN correlation_24h < 0.5 THEN 1 ELSE 0 END::SMALLINT AS corr_breakdown
FROM zone_correlations;
""")

print_table_status('zone_correlations')
print_table_status('neighbor_map')
print_table_status('zone_neighbor_correlation')

zone_correlations created (neighbor pairs only).
zone_correlations: exists, rows = 885,072
neighbor_map: exists, rows = 11
zone_neighbor_correlation: exists, rows = 885,072


## 9. Step 8 - Prediction features

Build features for next-hour load forecasting: lag values (1h, 24h, 168h), rolling means, historical hour/day-of-week averages, and a temperature proxy (rolling average of same-hour-same-DOW observations).

In [22]:
run_sql("""
DROP TABLE IF EXISTS prediction_features CASCADE;
CREATE TABLE prediction_features AS
WITH base AS (
    SELECT
        el.zone,
        el.datetime,
        el.load_mw AS current_load,
        tf.hour_of_day,
        tf.day_of_week,
        LAG(el.load_mw, 1)   OVER w AS load_lag_1h,
        LAG(el.load_mw, 24)  OVER w AS load_lag_24h,
        LAG(el.load_mw, 168) OVER w AS load_lag_168h,
        AVG(el.load_mw) OVER (
            PARTITION BY el.zone ORDER BY el.datetime
            ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
        ) AS pred_rolling_mean_6h,
        AVG(el.load_mw) OVER (
            PARTITION BY el.zone ORDER BY el.datetime
            ROWS BETWEEN 24 PRECEDING AND 1 PRECEDING
        ) AS pred_rolling_mean_24h,
        AVG(el.load_mw) OVER (
            PARTITION BY el.zone ORDER BY el.datetime
            ROWS BETWEEN 168 PRECEDING AND 1 PRECEDING
        ) AS pred_rolling_mean_168h
    FROM energy_load el
    JOIN temporal_features tf
      ON el.zone = tf.zone AND el.datetime = tf.datetime
    WINDOW w AS (PARTITION BY el.zone ORDER BY el.datetime)
),
hour_avg AS (
    SELECT
        el.zone,
        tf.hour_of_day,
        AVG(el.load_mw) AS hour_of_day_avg
    FROM energy_load el
    JOIN temporal_features tf
      ON el.zone = tf.zone AND el.datetime = tf.datetime
    GROUP BY el.zone, tf.hour_of_day
),
dow_avg AS (
    SELECT
        el.zone,
        tf.day_of_week,
        AVG(el.load_mw) AS day_of_week_avg
    FROM energy_load el
    JOIN temporal_features tf
      ON el.zone = tf.zone AND el.datetime = tf.datetime
    GROUP BY el.zone, tf.day_of_week
),
temp_proxy AS (
    SELECT
        zone,
        datetime,
        AVG(load_mw) OVER (
            PARTITION BY zone,
                         EXTRACT(HOUR FROM datetime)::int,
                         EXTRACT(DOW  FROM datetime)::int
            ORDER BY datetime
            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
        ) AS temp_proxy_load
    FROM energy_load
)
SELECT
    b.zone,
    b.datetime,
    b.current_load,
    b.load_lag_1h,
    b.load_lag_24h,
    b.load_lag_168h,
    b.pred_rolling_mean_6h,
    b.pred_rolling_mean_24h,
    b.pred_rolling_mean_168h,
    ha.hour_of_day_avg,
    da.day_of_week_avg,
    tp.temp_proxy_load
FROM base b
LEFT JOIN hour_avg ha
  ON b.zone = ha.zone AND b.hour_of_day = ha.hour_of_day
LEFT JOIN dow_avg da
  ON b.zone = da.zone AND b.day_of_week = da.day_of_week
LEFT JOIN temp_proxy tp
  ON b.zone = tp.zone AND b.datetime = tp.datetime;
""")
print_table_status('prediction_features')

prediction_features: exists, rows = 1,090,344


## 10. Step 9 - Consolidated table

Join all intermediate tables into `energy_consolidated` - one row per zone per hour with 30+ columns: load data, temporal features, rolling statistics, volatility regime, consumption pattern, neighbor correlation, prediction features, and an anomaly flag (correlation breakdown OR 3σ deviation).

In [23]:
run_sql("""
DROP TABLE IF EXISTS energy_consolidated CASCADE;
CREATE TABLE energy_consolidated AS
SELECT
    el.zone,
    el.datetime,
    el.load_mw,
    el.is_imputed,

    tf.hour_of_day,
    tf.day_of_week,
    tf.is_weekend,
    tf.is_business_hour,
    tf.month,
    tf.season,
    tf.is_peak,

    rs.rolling_mean_6h,
    rs.rolling_std_6h,
    rs.rolling_mean_24h,
    rs.rolling_std_24h,
    rs.rolling_mean_72h,
    rs.rolling_std_72h,
    rs.rolling_mean_168h,
    rs.rolling_std_168h,

    vr.daily_volatility,
    vr.volatility_regime,

    dc.consumption_pattern,

    znc.neighbor_zone,
    znc.correlation_24h AS neighbor_correlation_24h,
    znc.corr_breakdown,

    pf.current_load,
    pf.load_lag_1h,
    pf.load_lag_24h,
    pf.load_lag_168h,
    pf.pred_rolling_mean_6h,
    pf.pred_rolling_mean_24h,
    pf.pred_rolling_mean_168h,
    pf.hour_of_day_avg,
    pf.day_of_week_avg,
    pf.temp_proxy_load,

    CASE
        WHEN znc.corr_breakdown = 1 THEN 1
        WHEN rs.rolling_std_24h IS NOT NULL
         AND ABS(el.load_mw - rs.rolling_mean_24h) > 3 * rs.rolling_std_24h THEN 1
        ELSE 0
    END::SMALLINT AS anomaly_flag,

    TO_CHAR(el.datetime, 'YYYY-MM') AS year_month
FROM energy_load el
LEFT JOIN temporal_features tf
  ON el.zone = tf.zone AND el.datetime = tf.datetime
LEFT JOIN rolling_stats rs
  ON el.zone = rs.zone AND el.datetime = rs.datetime
LEFT JOIN volatility_regimes vr
  ON el.zone = vr.zone AND el.datetime::date = vr.day
LEFT JOIN daily_clusters dc
  ON el.zone = dc.zone AND el.datetime::date = dc.day
LEFT JOIN zone_neighbor_correlation znc
  ON el.zone = znc.zone AND el.datetime = znc.datetime
LEFT JOIN prediction_features pf
  ON el.zone = pf.zone AND el.datetime = pf.datetime;
""")

run_sql("CREATE INDEX IF NOT EXISTS idx_energy_consolidated_zone_dt ON energy_consolidated(zone, datetime);")
print_table_status('energy_consolidated')

energy_consolidated: exists, rows = 1,090,344


## 11. Final Verification

Confirm that all 15 project tables were created successfully.

In [24]:
for tbl in [
    'energy_load',
    'missing_hours',
    'gap_summary',
    'gap_info',
    'temporal_features',
    'rolling_stats',
    'volatility_regimes',
    'volatility_transitions',
    'daily_profiles',
    'daily_clusters',
    'zone_correlations',
    'neighbor_map',
    'zone_neighbor_correlation',
    'prediction_features',
    'energy_consolidated'
]:
    print_table_status(tbl)

energy_load: exists, rows = 1,090,344
missing_hours: exists, rows = 217
gap_summary: exists, rows = 12
gap_info: exists, rows = 217
temporal_features: exists, rows = 1,090,344
rolling_stats: exists, rows = 1,090,344
volatility_regimes: exists, rows = 45,443
volatility_transitions: exists, rows = 103
daily_profiles: exists, rows = 45,443
daily_clusters: exists, rows = 45,443
zone_correlations: exists, rows = 885,072
neighbor_map: exists, rows = 11
zone_neighbor_correlation: exists, rows = 885,072
prediction_features: exists, rows = 1,090,344
energy_consolidated: exists, rows = 1,090,344


In [25]:
run_sql("""
SELECT zone, COUNT(*) AS n_rows, MIN(datetime) AS first_ts, MAX(datetime) AS last_ts
FROM energy_load
GROUP BY zone
ORDER BY zone;
""", fetch=True)

,zone,n_rows,first_ts,last_ts
0,AEP,121296,2004-10-01 01:00:00,2018-08-03
1,COMED,66504,2011-01-01 01:00:00,2018-08-03
2,DAYTON,121296,2004-10-01 01:00:00,2018-08-03
3,DEOK,57744,2012-01-01 01:00:00,2018-08-03
4,DOM,116208,2005-05-01 01:00:00,2018-08-03
5,DUQ,119088,2005-01-01 01:00:00,2018-08-03
6,EKPC,45336,2013-06-01 01:00:00,2018-08-03
7,FE,62880,2011-06-01 01:00:00,2018-08-03
8,NI,58464,2004-05-01 01:00:00,2011-01-01
9,PJME,145392,2002-01-01 01:00:00,2018-08-03


In [26]:
run_sql("""
SELECT *
FROM gap_summary
ORDER BY zone;
""", fetch=True)

,zone,number_of_gaps,total_missing_hours,longest_gap_hours
0,AEP,27,27,1
1,COMED,11,11,1
2,DAYTON,25,25,1
3,DEOK,9,9,1
4,DOM,23,23,1
5,DUQ,24,24,1
6,EKPC,6,6,1
7,FE,10,10,1
8,NI,14,14,1
9,PJME,30,30,1


In [27]:
run_sql("""
SELECT *
FROM neighbor_map
ORDER BY zone;
""", fetch=True)

,zone,neighbor_zone,avg_corr
0,AEP,DAYTON,0.943818
1,COMED,FE,0.903888
2,DAYTON,FE,0.965563
3,DEOK,DAYTON,0.962329
4,DOM,PJME,0.927533
5,DUQ,FE,0.929663
6,EKPC,AEP,0.892500
7,FE,DAYTON,0.965563
8,NI,DUQ,0.899570
9,PJME,DUQ,0.929177


In [ ]:
run_sql("""
SELECT *
FROM energy_consolidated
ORDER BY zone, datetime
LIMIT 20;
""", fetch=True)

,zone,datetime,load_mw,is_imputed,hour_of_day,day_of_week,is_weekend,is_business_hour,month,season,...,load_lag_24h,load_lag_168h,pred_rolling_mean_6h,pred_rolling_mean_24h,pred_rolling_mean_168h,hour_of_day_avg,day_of_week_avg,temp_proxy_load,anomaly_flag,year_month
0,AEP,2004-10-01 01:00:00,12379.0,0,1,5,0,0,10,transition,...,None,None,NaN,NaN,NaN,13891.478433,15773.265437,None,0,2004-10
1,AEP,2004-10-01 02:00:00,11935.0,0,2,5,0,0,10,transition,...,None,None,12379.000000,12379.000000,12379.000000,13432.300950,15773.265437,None,0,2004-10
2,AEP,2004-10-01 03:00:00,11692.0,0,3,5,0,0,10,transition,...,None,None,12157.000000,12157.000000,12157.000000,13183.716066,15773.265437,None,0,2004-10
3,AEP,2004-10-01 04:00:00,11597.0,0,4,5,0,0,10,transition,...,None,None,12002.000000,12002.000000,12002.000000,13095.560645,15773.265437,None,0,2004-10
4,AEP,2004-10-01 05:00:00,11681.0,0,5,5,0,0,10,transition,...,None,None,11900.750000,11900.750000,11900.750000,13240.535813,15773.265437,None,0,2004-10
5,AEP,2004-10-01 06:00:00,12280.0,0,6,5,0,0,10,transition,...,None,None,11856.800000,11856.800000,11856.800000,13802.401464,15773.265437,None,0,2004-10
6,AEP,2004-10-01 07:00:00,13692.0,0,7,5,0,0,10,transition,...,None,None,11927.333333,11927.333333,11927.333333,14781.668381,15773.265437,None,0,2004-10
7,AEP,2004-10-01 08:00:00,14618.0,0,8,5,0,1,10,transition,...,None,None,12146.166667,12179.428571,12179.428571,15478.830233,15773.265437,None,0,2004-10
8,AEP,2004-10-01 09:00:00,14903.0,0,9,5,0,1,10,transition,...,None,None,12593.333333,12484.250000,12484.250000,15822.653740,15773.265437,None,0,2004-10
9,AEP,2004-10-01 10:00:00,15118.0,0,10,5,0,1,10,transition,...,None,None,13128.500000,12753.000000,12753.000000,16084.283934,15773.265437,None,0,2004-10
